# Assignment 03 - Predicting Used Car Price
### EECE 5644 - Summer 2026

### Noah Caney and Aakarsh Arora

Predict the fair market price of a used car from its features using Linear Regression,
then use Lasso (L1) and Ridge (L2) regularization. Same steps we used on the Melbourne housing dataset.

### Load the data
In Colab, upload `car_details_v4.csv` using the folder icon on the left first.

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Suppress warnings for clean notebook
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# read dataset
dataset = pd.read_csv('car_details_v4.csv')
dataset.columns = dataset.columns.str.strip()

In [ ]:
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset.nunique()

#### Clean up the engine columns
Engine, Max Power and Max Torque are stored as text like "1198 cc" and "87 bhp @ 6000 rpm".
We just want the number, so we pull out the first number from each and turn it into a float.

In [ ]:
dataset['Engine']     = dataset['Engine'].str.extract(r'(\d+\.?\d*)').astype(float)
dataset['Max Power']  = dataset['Max Power'].str.extract(r'(\d+\.?\d*)').astype(float)
dataset['Max Torque'] = dataset['Max Torque'].str.extract(r'(\d+\.?\d*)').astype(float)

dataset[['Engine', 'Max Power', 'Max Torque']].head()

#### Let's use limited columns which makes more sense for serving our purpose
(We leave out Model because it has over 1000 different values, which would create too many columns.)

In [ ]:
cols_to_use = ['Make', 'Year', 'Kilometer', 'Fuel Type', 'Transmission', 'Location',
               'Color', 'Owner', 'Seller Type', 'Engine', 'Max Power', 'Max Torque',
               'Drivetrain', 'Length', 'Width', 'Height', 'Seating Capacity',
               'Fuel Tank Capacity', 'Price']
dataset = dataset[cols_to_use]

In [ ]:
dataset.head()

In [ ]:
dataset.shape

#### Checking for Nan values

In [ ]:
dataset.isna().sum()

#### Handling Missing values

In [ ]:
# fill the numeric columns with the mean
num_cols = ['Engine', 'Max Power', 'Max Torque', 'Length', 'Width',
            'Height', 'Seating Capacity', 'Fuel Tank Capacity']
dataset[num_cols] = dataset[num_cols].fillna(dataset[num_cols].mean())

# fill the categorical column with the most common value
dataset['Drivetrain'] = dataset['Drivetrain'].fillna(dataset['Drivetrain'].mode()[0])

**Drop NA values of Price, since it's our predictive variable we won't impute it**

In [ ]:
dataset.dropna(inplace=True)

In [ ]:
dataset.shape

#### Let's one hot encode the categorical features

In [ ]:
dataset = pd.get_dummies(dataset, drop_first=True)

In [ ]:
dataset.head()

#### Let's bifurcate our dataset into train and test dataset

In [ ]:
X = dataset.drop('Price', axis=1)
y = dataset['Price']

In [ ]:
from sklearn.model_selection import train_test_split
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=2)

#### Let's train our Linear Regression Model on training dataset and check the accuracy on test set

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(train_X, train_y)

In [ ]:
reg.score(test_X, test_y)

In [ ]:
reg.score(train_X, train_y)

#### Evaluate with RMSE, MAE and R2 Score
The three metrics from the evaluation sheet. MAE and RMSE are in the same units as Price.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = reg.predict(test_X)

# RMSE
rmse = np.sqrt(mean_squared_error(test_y, y_pred))
print("RMSE:", rmse)

# MAE
mae = mean_absolute_error(test_y, y_pred)
print("MAE:", mae)

# R2 Score
r2 = r2_score(test_y, y_pred)
print("R2 Score:", r2)

In [ ]:
# actual vs predicted on a sample of the test cars
results = test_X.copy()
results['actual']    = test_y.values
results['predicted'] = y_pred.round(2)
results['error']     = (results['actual'] - results['predicted']).round(2)
results[['actual', 'predicted', 'error']].head(10)

**The training score is higher than the test score, which means the model is overfitting a bit.
Let's try Lasso and Ridge regression.**

#### Using Lasso (L1 Regularized) Regression Model

In [ ]:
from sklearn import linear_model
lasso_reg = linear_model.Lasso(alpha=50, max_iter=100, tol=0.1)
lasso_reg.fit(train_X, train_y)

In [ ]:
lasso_reg.score(test_X, test_y)

In [ ]:
lasso_reg.score(train_X, train_y)

#### Using Ridge (L2 Regularized) Regression Model

In [ ]:
from sklearn.linear_model import Ridge
ridge_reg = Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_reg.fit(train_X, train_y)

In [ ]:
ridge_reg.score(test_X, test_y)

In [ ]:
ridge_reg.score(train_X, train_y)

**Linear Regression overfits a little (training score higher than test score). Lasso and Ridge
add a penalty to reduce overfitting so the model generalizes better to new cars, which is what we
studied in the regularization slides. This model could help WheelsBazaar flag overpriced or
underpriced listings and give the instant-buy program an automatic price estimate.**